# ExoData Challenge - Fase 1: Ingesta y Exploración
## Tarea 1.1: Cargar y auditar el dataset PSCompPars

In [1]:
# CELDA 1 - Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Python version: 3.14.4 (main, Apr  7 2026, 13:13:20) [GCC 12.3.0]
Pandas version: 3.0.2
NumPy version: 2.4.4


In [5]:
# CELDA 2 - Leer CSV
df = pd.read_csv('../data/PSCompPars_2026.csv', comment='#')

print(f"Shape: {df.shape}")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Shape: (6160, 320)
Filas: 6160
Columnas: 320


/tmp/ipykernel_50518/440543180.py:2: DtypeWarning: Columns (0: pl_imppar_reflink, 1: pl_trandep_reflink, 2: pl_trandur_reflink, 3: pl_ratror_reflink, 4: pl_occdep_reflink, 5: pl_projobliq_reflink, 6: pl_trueobliq_reflink) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/PSCompPars_2026.csv', comment='#')


In [6]:
# CELDA 3 - Inspección
print("=== PRIMERA FILA ===")
df.head(1).T

print("=== ÚLTIMA FILA ===")
df.tail(1).T

print("=== TIPOS DE DATOS ===")
df.dtypes.head(10)

=== PRIMERA FILA ===
=== ÚLTIMA FILA ===
=== TIPOS DE DATOS ===


rowid          int64
pl_name          str
hostname         str
pl_letter        str
hd_name          str
hip_name         str
tic_id           str
gaia_dr2_id      str
gaia_dr3_id      str
sy_snum        int64
dtype: object

In [4]:
# CELDA 4 - Completitud por columna
completitud = df.notna().mean().sort_values(ascending=False)

reporte = pd.DataFrame({
    'completitud': completitud,
    'missing_pct': (1 - completitud) * 100
})

top_20 = reporte.head(20)
print("=== TOP 20 COLUMNAS MÁS COMPLETAS ===")
display(top_20)

top_20.to_csv('docs/top20_completitud.csv', index=True)
print("\n✓ Reporte guardado en docs/top20_completitud.csv")

=== TOP 20 COLUMNAS MÁS COMPLETAS ===


,completitud,missing_pct
pl_letter,1.000000,0.000000
hostname,1.000000,0.000000
rowid,1.000000,0.000000
pl_name,1.000000,0.000000
sy_mnum,1.000000,0.000000
cb_flag,1.000000,0.000000
discoverymethod,1.000000,0.000000
disc_refname,1.000000,0.000000
sy_snum,1.000000,0.000000
sy_pnum,1.000000,0.000000



✓ Reporte guardado en docs/top20_completitud.csv


In [5]:
# CELDA 5 - Variables CRÍTICAS
critical_vars = ['pl_rade', 'pl_bmasse', 'pl_orbper', 'pl_orbsmax', 'pl_eqt', 'st_teff', 'st_met', 'sy_pnum', 'discoverymethod']

print("=== COMPLETITUD DE VARIABLES CRÍTICAS ===")
for var in critical_vars:
    pct = df[var].notna().mean() * 100
    print(f"{var:<20}: {pct:>6.1f}%")

df[critical_vars].notna().mean().to_csv('docs/critical_vars_completitud.csv')
print("\n✓ Reporte guardado")

=== COMPLETITUD DE VARIABLES CRÍTICAS ===
pl_rade             :   99.0%
pl_bmasse           :   99.3%
pl_orbper           :   94.4%
pl_orbsmax          :   94.7%
pl_eqt              :   74.5%
st_teff             :   94.9%
st_met              :   90.6%
sy_pnum             :  100.0%
discoverymethod     :  100.0%

✓ Reporte guardado


In [6]:
# CELDA 6 - Detectar outliers
for var in ['pl_orbper', 'pl_rade', 'pl_bmasse']:
    if var in df.columns:
        q99 = df[var].quantile(0.999)
        q01 = df[var].quantile(0.001)
        extreme_high = df[df[var] > q99].shape[0]
        extreme_low = df[df[var] < q01].shape[0]
        print(f"{var}: {extreme_high} valores > q99, {extreme_low} valores < q01")

pl_orbper: 6 valores > q99, 6 valores < q01
pl_rade: 7 valores > q99, 7 valores < q01
pl_bmasse: 7 valores > q99, 7 valores < q01


In [7]:
# CELDA 7 - Distribución por método de detección
method_counts = df['discoverymethod'].value_counts()

print("=== DISTRIBUCIÓN POR MÉTODO DE DETECCIÓN ===")
print(method_counts.head(10))
print(f"\nTotal de planetas con method conocido: {method_counts.sum()} de {len(df)}")

method_counts.to_csv('docs/detection_method_distribution.csv')
print("\n✓ Reporte de sesgo guardado")

=== DISTRIBUCIÓN POR MÉTODO DE DETECCIÓN ===
discoverymethod
Transit                          4523
Radial Velocity                  1180
Microlensing                      278
Imaging                            96
Transit Timing Variations          40
Eclipse Timing Variations          17
Orbital Brightness Modulation       9
Pulsar Timing                       8
Astrometry                          6
Pulsation Timing Variations         2
Name: count, dtype: int64

Total de planetas con method conocido: 6160 de 6160

✓ Reporte de sesgo guardado


In [8]:
# CELDA 8 - Resumen ejecutivo
print("\n" + "="*60)
print("RESUMEN DE CALIDAD DE DATOS")
print("="*60)
print(f"✓ Total planetas: {len(df):,}")
print(f"✓ Total variables: {len(df.columns)}")
print(f"✓ Variables con >70% completitud: {(df.notna().mean() > 0.7).sum()}")
print(f"✓ Variables con >85% completitud: {(df.notna().mean() > 0.85).sum()}")
print("\n✓✓✓ DATASET CARGADO Y AUDITADO ✓✓✓")
print("\nSiguiente: Tarea 1.2 - Reporte de valores nulos detallado")


RESUMEN DE CALIDAD DE DATOS
✓ Total planetas: 6,160
✓ Total variables: 320
✓ Variables con >70% completitud: 215
✓ Variables con >85% completitud: 171

✓✓✓ DATASET CARGADO Y AUDITADO ✓✓✓

Siguiente: Tarea 1.2 - Reporte de valores nulos detallado
